# Lab 02.1 — DataFrame e Spark SQL

Un **DataFrame** è una collezione distribuita di righe con uno **schema** (colonne con nome e tipo), come una tabella.
Spark offre due linguaggi equivalenti per interrogarlo:

* la **API DataFrame** (Python, Scala, Java, R);
* **SQL** standard (ANSI), tramite *viste temporanee* o tabelle.

Entrambi producono lo stesso **piano logico**, ottimizzato dal motore **Catalyst** ed eseguito da **Tungsten** (codice generato a runtime, memoria off-heap).
Il dataset è un e-commerce sintetico generato in parallelo dagli executor (`dati_shop.py`).

In [ ]:
import os, time
from pyspark.sql import SparkSession, functions as F, Window
import dati_shop

spark = (SparkSession.builder
         .appName("Lab02-DataFrame-SQL")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.sql.shuffle.partitions", "8")   # default 200: troppe per un piccolo cluster
         .getOrCreate())
print("Spark", spark.version, "| master:", spark.sparkContext.master)

## 1. Creazione dei DataFrame e schema

In [ ]:
# Al primo avvio genera i dati in parallelo e li salva in Parquet (data/shop/)
tabelle  = dati_shop.carica(spark, n_ordini=2_000_000)
clienti, prodotti, ordini = tabelle["clienti"], tabelle["prodotti"], tabelle["ordini"]

ordini.printSchema()
ordini.show(5)
print(f"ordini: {ordini.count():,}  clienti: {clienti.count():,}  prodotti: {prodotti.count()}")

In [ ]:
# Statistiche descrittive calcolate in parallelo
ordini.select("quantita", "data").summary("count", "min", "max", "mean").show()

## 2. API DataFrame
Trasformazioni *lazy* (`select`, `where`, `withColumn`, `groupBy`, `join`...) e azioni (`show`, `count`, `collect`, `write`...).

In [ ]:
righe = (ordini
         .join(prodotti, "prodotto_id")                                   # join sulla chiave comune
         .withColumn("importo", F.col("quantita") * F.col("prezzo"))      # colonna calcolata
         .withColumn("mese", F.date_trunc("month", "data").cast("date")))

fatturato_categoria = (righe
    .where(F.year("data") == 2024)
    .groupBy("categoria")
    .agg(F.count("*").alias("ordini"),
         F.sum("importo").alias("fatturato"),
         F.round(F.avg("quantita"), 2).alias("q_media"))
    .orderBy(F.desc("fatturato")))

fatturato_categoria.show()

## 3. La stessa interrogazione in SQL
Registriamo i DataFrame come **viste temporanee**: diventano tabelle interrogabili con SQL nella sessione corrente.

In [ ]:
clienti.createOrReplaceTempView("clienti")
prodotti.createOrReplaceTempView("prodotti")
ordini.createOrReplaceTempView("ordini")

sql_df = spark.sql("""
    SELECT p.categoria,
           COUNT(*)                         AS ordini,
           SUM(o.quantita * p.prezzo)       AS fatturato,
           ROUND(AVG(o.quantita), 2)        AS q_media
    FROM ordini o
    JOIN prodotti p USING (prodotto_id)
    WHERE year(o.data) = 2024
    GROUP BY p.categoria
    ORDER BY fatturato DESC
""")
sql_df.show()

In [ ]:
# SQL e DataFrame sono solo due "front-end": Catalyst produce lo stesso piano fisico
sql_df.explain(mode="formatted")

## 4. Join tra più tabelle e aggregazioni multidimensionali

In [ ]:
spark.sql("""
    SELECT c.citta, o.canale, COUNT(*) AS ordini, ROUND(SUM(o.quantita * p.prezzo), 0) AS fatturato
    FROM ordini o
    JOIN clienti  c ON o.cliente_id  = c.cliente_id
    JOIN prodotti p ON o.prodotto_id = p.prodotto_id
    GROUP BY c.citta, o.canale
    ORDER BY c.citta, o.canale
""").show(12)

In [ ]:
# ROLLUP: subtotali per città e totale generale (righe con NULL)
spark.sql("""
    SELECT c.citta, o.canale, COUNT(*) AS ordini
    FROM ordini o JOIN clienti c USING (cliente_id)
    WHERE c.citta IN ('Roma', 'Milano')
    GROUP BY ROLLUP(c.citta, o.canale)
    ORDER BY c.citta NULLS LAST, o.canale NULLS LAST
""").show()

In [ ]:
# Tabella pivot: fatturato per categoria (righe) e anno (colonne)
(righe.withColumn("anno", F.year("data"))
      .groupBy("categoria").pivot("anno", [2024, 2025]).agg(F.round(F.sum("importo"), 0))
      .orderBy("categoria").show())

## 5. Funzioni finestra (window functions)
Calcolano un valore per ogni riga guardando un insieme di righe "vicine" (la finestra), **senza** collassare le righe come fa `GROUP BY`.

In [ ]:
# Top 3 prodotti per fatturato in ogni categoria
spark.sql("""
    WITH vendite AS (
        SELECT p.categoria, p.prodotto, SUM(o.quantita * p.prezzo) AS fatturato
        FROM ordini o JOIN prodotti p USING (prodotto_id)
        GROUP BY p.categoria, p.prodotto
    )
    SELECT * FROM (
        SELECT *, RANK() OVER (PARTITION BY categoria ORDER BY fatturato DESC) AS posizione
        FROM vendite
    ) WHERE posizione <= 3
    ORDER BY categoria, posizione
""").show(12)

In [ ]:
# Andamento mensile: totale cumulato e variazione rispetto al mese precedente (API DataFrame)
# (finestra senza PARTITION BY: pochi dati aggregati, Spark avvisa che userà una sola partizione)
mensile = righe.groupBy("mese").agg(F.round(F.sum("importo"), 0).alias("fatturato"))
w = Window.orderBy("mese")
(mensile
   .withColumn("cumulato", F.sum("fatturato").over(w.rowsBetween(Window.unboundedPreceding, 0)))
   .withColumn("mese_prec", F.lag("fatturato").over(w))
   .withColumn("variazione_%", F.round((F.col("fatturato") / F.col("mese_prec") - 1) * 100, 1))
   .orderBy("mese").show(8))

## 6. Funzioni definite dall'utente (UDF)
Le funzioni **built-in** girano nella JVM e sono ottimizzate da Catalyst.
Una **UDF Python** richiede di serializzare ogni riga verso un processo Python: è più lenta.
Le **Pandas UDF** (vettoriali) trasferiscono blocchi di righe in formato **Apache Arrow**: molto più efficienti.

In [ ]:
import pandas as pd
from pyspark.sql.functions import udf, pandas_udf

@udf("double")
def iva_udf(x):
    return round(float(x) * 1.22, 2)

@pandas_udf("double")
def iva_pandas(x: pd.Series) -> pd.Series:
    return (x.astype("float64") * 1.22).round(2)

base = righe.select(F.col("importo").cast("double").alias("importo"))

for nome, espr in [("built-in", F.col("importo") * 1.22),
                   ("UDF Python", iva_udf("importo")),
                   ("Pandas UDF", iva_pandas("importo"))]:
    t0 = time.time()
    base.select(F.sum(espr)).collect()
    print(f"{nome:12s}: {time.time() - t0:.2f}s")

In [ ]:
# Le UDF si possono registrare e usare anche in SQL
spark.udf.register("iva", iva_pandas)
spark.sql("SELECT prodotto, prezzo, iva(prezzo) AS prezzo_ivato FROM prodotti LIMIT 5").show()

## 7. Catalyst: dal SQL al piano fisico
`explain(extended=True)` mostra le quattro fasi: piano **analizzato** → **ottimizzato** → **fisico**.
Osserva come il filtro sulla città viene spostato **prima** del join (*predicate pushdown*) e come vengono lette solo le colonne necessarie (*column pruning*).

In [ ]:
q = spark.sql("""
    SELECT c.nome, o.data
    FROM ordini o JOIN clienti c USING (cliente_id)
    WHERE c.citta = 'Bari' AND o.quantita > 4
""")
q.explain(extended=True)

Nel piano fisico cerca:
* `PushedFilters: [IsNotNull(citta), EqualTo(citta,Bari)]` → il filtro è passato al lettore Parquet;
* `ReadSchema` → vengono lette solo le colonne usate;
* `BroadcastHashJoin` → la tabella clienti (piccola) è inviata a tutti gli executor invece di fare uno shuffle.

In [ ]:
spark.stop()